# Credit Scoring - Feature Engineering
Only deterministic transformations are applied here. Imputation and outlier capping depend on the data distribution and will be part of the modelling pipeline, fitted on the training set only.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

df = pd.read_csv("../data/cs-training.csv", index_col=0)

## Removing errors from the data

In [2]:
rows_before = df.shape[0]

df = df[df["age"] > 0]

print("Number of deleted records:", rows_before - df.shape[0])

Number of deleted records: 1


## Invalid values correction

### DebtRatio

In [3]:
df.loc[(df["MonthlyIncome"].isna()) | (df["MonthlyIncome"] == 0), "DebtRatio"] = np.nan
print(df["DebtRatio"].quantile([0.5, 0.9, 0.95, 0.99, 0.999, 1.0]))

0.500       0.292
0.900       0.725
0.950       0.982
0.990       2.994
0.999    1467.151
1.000   61106.500
Name: DebtRatio, dtype: float64


### Special codes

In [4]:
cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

for c in cols:
    print(
        f"{c}: number of replaced values = {(df[c] > 90).sum()} ({(df[c] > 90).mean():.1%})"
    )
    df.loc[df[c] > 90, c] = -1


NumberOfTime30-59DaysPastDueNotWorse: number of replaced values = 269 (0.2%)
NumberOfTime60-89DaysPastDueNotWorse: number of replaced values = 269 (0.2%)
NumberOfTimes90DaysLate: number of replaced values = 269 (0.2%)


## Capping delinquency counters

In [5]:
cap_dict = {
    "NumberOfTime30-59DaysPastDueNotWorse": 6,
    "NumberOfTime60-89DaysPastDueNotWorse": 4,
    "NumberOfTimes90DaysLate": 4,
}

for key in cap_dict.keys():
    print(
        f"{key}: number of capped values = {(df[key] > cap_dict[key]).sum()} ({(df[key] > cap_dict[key]).mean():.1%})"
    )
    df[key] = df[key].clip(upper=cap_dict[key])

NumberOfTime30-59DaysPastDueNotWorse: number of capped values = 99 (0.1%)
NumberOfTime60-89DaysPastDueNotWorse: number of capped values = 63 (0.0%)
NumberOfTimes90DaysLate: number of capped values = 313 (0.2%)


## New features
In the new delinquency based features, records with special status receive the value -1, consistently with the original counters.

### IsIncomeMissing

In [6]:
df["IsIncomeMissing"] = df["MonthlyIncome"].isna().astype(int)

### IsSpecialStatus

In [7]:
# IsSpecialStatus calculated based on one of variables (based on EDA special codes are in all three variables simultaneously)
df["IsSpecialStatus"] = (df["NumberOfTime30-59DaysPastDueNotWorse"] == -1).astype(int)

### IncomePerDependent

In [8]:
df["IncomePerDependent"] = df["MonthlyIncome"] / (1 + df["NumberOfDependents"])

print("Number of missing values in MonthlyIncome:", df["MonthlyIncome"].isna().sum())
print(
    "Number of missing values in NumberOfDependents:",
    df["NumberOfDependents"].isna().sum(),
)
print(
    "Number of missing values in MonthlyIncome and NumberOfDependents:",
    (df["NumberOfDependents"].isna() & df["MonthlyIncome"].isna()).sum(),
)
print(
    "Number of missing values in IncomePerDependent:",
    df["IncomePerDependent"].isna().sum(),
)

Number of missing values in MonthlyIncome: 29731
Number of missing values in NumberOfDependents: 3924
Number of missing values in MonthlyIncome and NumberOfDependents: 3924
Number of missing values in IncomePerDependent: 29731


Missing values in `NumberOfDependents` are a subset of missing `MonthlyIncome`, so the new feature has the same number of missing values as income.

### TotalNumberPastDue

In [9]:
# Variable calculated on values after capping
sum_past_due = (
    df["NumberOfTime30-59DaysPastDueNotWorse"]
    + df["NumberOfTime60-89DaysPastDueNotWorse"]
    + df["NumberOfTimes90DaysLate"]
)
df["TotalNumberPastDue"] = np.where(
    df["NumberOfTime30-59DaysPastDueNotWorse"] == -1, -1, sum_past_due
)

### HasAnyDelinquency

In [10]:
df["HasAnyDelinquency"] = np.where(
    df["TotalNumberPastDue"] == -1, -1, (df["TotalNumberPastDue"] > 0).astype(int)
)

## Final check

In [11]:
print(f"Shape: {df.shape}")
print("\nMissing values:")
print(df.isna().sum())
print("\nNew features summary:")
display(
    df[
        [
            "IsIncomeMissing",
            "IsSpecialStatus",
            "IncomePerDependent",
            "TotalNumberPastDue",
            "HasAnyDelinquency",
        ]
    ].describe()
)

Shape: (149999, 16)

Missing values:
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                               31365
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
IsIncomeMissing                             0
IsSpecialStatus                             0
IncomePerDependent                      29731
TotalNumberPastDue                          0
HasAnyDelinquency                           0
dtype: int64

New features summary:


,IsIncomeMissing,IsSpecialStatus,IncomePerDependent,TotalNumberPastDue,HasAnyDelinquency
count,149999.000,149999.000,120268.000,149999.000,149999.000
mean,0.198,0.002,4509.256,0.391,0.199
std,0.399,0.042,8884.052,1.051,0.404
min,0.000,0.000,0.000,-1.000,-1.000
25%,0.000,0.000,1960.000,0.000,0.000
50%,0.000,0.000,3333.333,0.000,0.000
75%,0.000,0.000,5658.000,0.000,0.000
max,1.000,1.000,1794060.000,14.000,1.000


Leaving missing values in `DebtRatio`, `MonthlyIncome`, `NumberOfDependents` and `IncomePerDependent` is intentional - they will be imputed in the modelling pipeline.

`IncomePerDependent` will also be capped at the 99th percentie in the pipeline, because of the visible outliers (maximum value 1 794 060).

## Save processed data

In [12]:
df.to_parquet("../data/processed.parquet")